In [ ]:
##############
# Homework 3 #
##############


###################
# Read This First #
###################


# All functions that you need to modify are marked with 'EXERCISE' in their header comments.
# Do not modify astar.py
# This file also contains many helper functions. You may call any of them in your functions.


# Due to the memory limitation, the A* algorithm may crash on some hard sokoban problems if too many
# nodes are generated. Improving the quality of the heuristic will mitigate
# this problem, as it will allow A* to solve hard problems with fewer node expansions.


# Remember that most functions are not graded on efficiency (only correctness).
# Efficiency can only influence your heuristic performance in the competition (which will affect your score).


# Load the astar.py and do not modify it.
import astar
# Load the numpy package and the state is represented as a numpy array during this homework.
import numpy as np


# a_star perform the A* algorithm with the start_state (numpy array), goal_test (function), successors (function) and
# heuristic (function). a_star prints the solution from start_state to goal_state (path), calculates the number of
# generated nodes (node_generated) and expanded nodes (node_expanded), and the solution depth (len(path)-1). a_star
# also provides the following functions for printing states and moves: prettyMoves(path): Translate the solution to a
# list of moves printlists(path): Visualize the solution and Print a list of states
def a_star(start_state, goal_test, successors, heuristic):
    goal_node, node_generated, node_expanded = astar.a_star_search(start_state, goal_test, successors, heuristic)
    if goal_node:
        node = goal_node
        path = [node.state1]
        while node.parent:
            node = node.parent
            path.append(node.state1)
        path.reverse()

        # print('My path:{}'.format(path))
        # print(prettyMoves(path))
        # printlists(path)
        print('Nodes Generated by A*: {}'.format(node_generated))
        print('Nodes Expanded by A*: {}'.format(node_expanded))
        print('Solution Depth: {}'.format(len(path) - 1))
    else:
        print('no solution found')


# A shortcut function
# Transform the input state to numpy array. For other functions, the state s is presented as a numpy array.
# Goal-test and next-states stay the same throughout the assignment
# You can just call sokoban(init-state, heuristic function) to test the result
def sokoban(s, h):
    return a_star(np.array(s), goal_test, next_states, h)


# Define some global variables
blank = 0
wall = 1
box = 2
keeper = 3
star = 4
boxstar = 5
keeperstar = 6


# Some helper functions for checking the content of a square
def isBlank(v):
    return (v == blank)


def isWall(v):
    return (v == wall)


def isBox(v):
    return (v == box)


def isKeeper(v):
    return (v == keeper)


def isStar(v):
    return (v == star)


def isBoxstar(v):
    return (v == boxstar)


def isKeeperstar(v):
    return (v == keeperstar)


# Help function for get KeeperPosition
# Given state s (numpy array), return the position of the keeper by row, col
# The top row is the zeroth row
# The first (right) column is the zeroth column
def getKeeperPosition(s):
    row = s.shape[0]
    col = s.shape[1]
    for i in range(row):
        for j in range(col):
            if (isKeeper(s[i, j]) or isKeeperstar(s[i, j])):
                return i, j


# For input list s_list, remove all None element
# For example, if s_list = [1, 2, None, 3], returns [1, 2, 3]
def cleanUpList(s_list):
    clean = []
    for state in s_list:
        if state is not None:
            clean.append(state)
    return clean


# EXERCISE: Modify this function to return Ture
# if and only if s (numpy array) is a goal state of a Sokoban game.
# (no box is on a non-goal square)
# Remember, the number of goal can be larger than the number of box.
def goal_test(s):
    '''
    Given s, a numpy array, this function returns if the goal state of the game is reached.

    Input: s, numpy array with current state

    Output: True if goal state reached, False otherwise
    '''
    # if any spots are just a box (not on goal), then we are not at the goal state
    return not any(isBox(s[i,j]) for i in range(s.shape[0]) for j in range(s.shape[1]))


# EXERCISE: Modify this function to return the list of
# successor states of s (numpy array).
#
# This is the top-level next-states (successor) function.
# Some skeleton code is provided below.
# You may delete them totally, depending on your approach.
#
# If you want to use it, you will need to set 'result' to be
# the set of states after moving the keeper in each of the 4 directions.
#
# You can define the function try-move and decide how to represent UP,DOWN,LEFT,RIGHT.
# Any None result in the list can be removed by cleanUpList.
#
# When generated the successors states, you may need to copy the current state s (numpy array).
# A shallow copy (e.g, direcly set s1 = s) constructs a new compound object and then inserts references
# into it to the objects found in the original. In this case, any change in the numpy array s1 will also affect
# the original array s. Thus, you may need a deep copy (e.g, s1 = np.copy(s)) to construct an indepedent array.
def get_square(S,r,c):
  '''
  A function which given a state S, row number r, and column number c, returns
  the integer content of state S at square (r,c).

  If the square is outside the scope of the problem return the value of a wall.

  Input:
  - S, numpy array, state of game
  - r, integer, row on board
  - c, integer, column in board

  Returns:
  - integer, content of state S are r,c or a wall if not in scope
  '''
  # if invalid coordinates, return wall
  if r < 0 or r >= S.shape[0] or c < 0 or c >= S.shape[1]:
    return wall
  # otherwise return content of r,c
  else:
    return S[r,c]

def set_square(S,r,c,v):
  '''
  A function which given a state S, row number r, column number c, and square
  content v, returns a new state S', where (r,c) is set to value v.

  Input:
  - S, numpy array, state of game
  - r, integer, row on board
  - c, integer, column in board
  - v, integer, content of square

  Returns:
  - numpy array, where (r,c) is set to value v
  '''
  # if invalid coordinates, return the current state with no changes
  if r < 0 or r >= S.shape[0] or c < 0 or c >= S.shape[1]:
    return S
  # return S with the new changed value
  else:
    s_copy = np.copy(S)
    s_copy[r,c] = v
    return s_copy

def clear_keeper_old_tile(S,krow,kcol):
  '''
  A function which given a state and keeper's current row and col,
  clears the old tile. If it was previously just the keeper it is blank, and
  if it was keeper and star it is star.

  Input:
  - S, numpy array, state of game
  - krow, integer, row of keeper
  - kcol, integer, col of keeper

  Returns:
  - state with keeper's old tile replaced

  '''
  # clear keeper old tile to star if prev keeperstar
  if isKeeperstar(get_square(S,krow,kcol)):
    S = set_square(S,krow,kcol,star)
  # clear keeper old tile to blank if prev keeper
  else:
    S = set_square(S,krow,kcol,blank)
  return S

def try_move(S,D):
  '''
  A function which given S, the current state of the game, and D, a direction to
  move in, returns the state that is the result of moving the keeper in state S
  in direction D. If the move is invalid, returns None.

  U --> UP
  D --> DOWN
  L --> LEFT
  R --> RIGHT

  Input:
  - S, numpy array, state of game
  - D, direction to move keeper in

  Returns:
  - numpy array of the state after the move or None is invalid move

  '''
  krow, kcol = getKeeperPosition(S)

  # to use to calculate where we are trying to move keeper
  # nr, nc corresponds to next row/next col
  if D == 'U':
    nr,nc = -1,0
  elif D == 'D':
    nr,nc = 1,0
  elif D == 'L':
    nr,nc = 0,-1
  elif D == 'R':
    nr,nc = 0,1
  # if not a valid move, return None
  else:
    return None

  # calculate where keeper wants to go
  r,c = krow+nr, kcol+nc

  # coord for pushing a box (2 spots over in the specific direction)
  rr,cc = krow+2*nr, kcol+2*nc

  # square keeper wants to go to
  end = get_square(S,r,c)

  # if end goal is wall, return None - invalid move
  if isWall(end):
    return None

  # if blank or goal just move the keeper
  if isBlank(end) or isStar(end):
    # move keeper
    S = set_square(S,r,c, keeper if isBlank(end) else keeperstar)
    # clear old keeper tile
    S = clear_keeper_old_tile(S,krow,kcol)
    return S

  # if box or box star, have to check if you can move the box
  elif isBox(end) or isBoxstar(end):
    push_end = get_square(S,rr,cc)
    # if push spot is not blank or a star, cannot move there
    if not (isBlank(push_end) or isStar(push_end)):
      return None
    # clear old keeper tile
    S = clear_keeper_old_tile(S,krow,kcol)
    # remove box and move keeper
    S = set_square(S,r,c,keeperstar if isBoxstar(end) else keeper)
    # update box's new spot
    S = set_square(S,rr,cc,boxstar if isStar(push_end) else box)

    return S

  return None

def next_states(S):
  '''
  A function which take a state and returns the list of all states that can
  be reached from the given state in one move.

  Input:
  - s, numpy array, state of the game

  Returns:
  - list of states that can be reached from the given state in one move
  '''
  # get keeper's current square
  row, col = getKeeperPosition(S)

  # list to store possible states
  s_list = []

  # try each direction
  s_list.append(try_move(np.copy(S),'U'))
  s_list.append(try_move(np.copy(S),'D'))
  s_list.append(try_move(np.copy(S),'L'))
  s_list.append(try_move(np.copy(S),'R'))

  # remove Nones
  return cleanUpList(s_list)


# EXERCISE: Modify this function to compute the trivial
# admissible heuristic.
def h0(s):
    return 0

# EXERCISE: Modify this function to compute the
# number of misplaced boxes in state s (numpy array).
def h1(s):
  '''
  A heuristic function that given a state, s, returns the number of boxes which
  are not on goal positions.

  Input:
  - s, numpy array, state of game

  Output:
  - integer, number of boxes not on goal positions
  '''
  count = 0

  row = s.shape[0]
  col = s.shape[1]

  # loop through all spots
  for i in range(row):
      for j in range(col):
        # if it is a box (not boxstar or another integer), increment count
        if isBox(s[i,j]):
          count +=1

  return count

# EXERCISE: Is this heuristic admissible? Return true if yes, false if no. Explain your reasoning
# as comments in this function.
def h1_admissible():
  '''
  Yes, this heuristic is admissible. We know this because it never overestimates
  the true cost, or moves required. We are counting how many boxes are not on
  goals, and each one would need to be moved at LEAST once to be on a goal.
  This is either the actual cost to win the game or less, so the heuristic is
  admissible.

  '''
  return True

# EXERCISE:
# This function will be tested in various hard examples.
# Objective: make A* solve problems as fast as possible.
def h2(s):
  '''
  A heuristic function that given a state s, returns an integer greater than
  or equal to 0 to approximate the cost of the solution.

  I use the sum of the Manhattan distance from each unmatched box to
  its closest goal. I also add the distance the keeper will have to move to move
  the closest box.

  This heuristic is admissible since it does not overestimate the true cost to
  get all the boxes to goals. With each box, we calculate the Manhattan distance
  to the CLOSEST goal, not considering other obstacles and walls and boxes
  having the same closest goals. Thus, this is a lower bound. At minimum,
  the boxes would need that many pushes to get to the closest goal. We also
  calculate the Manhattan distance from keeper to closest box, but the keeper
  might have to also go around obstacles, so this is also an underestimate.

  Both components are the minimum cost that would be needed.

  Input:
  - s, numpy array, state of game

  Returns:
  - integer, approximate cost of solution
  '''
  # calculate coordinates of boxes and goals NOT matched already (only Box, not Boxstar)
  boxes = [(r,c) for r in range(s.shape[0]) for c in range(s.shape[1]) if isBox(s[r,c])]
  stars = [(r,c) for r in range(s.shape[0]) for c in range(s.shape[1]) if isStar(s[r,c])]

  total_cost = 0

  # if one is empty return 0 - no unmatched boxes or goals left
  if not boxes or not stars:
    return total_cost

  # loop through boxes
  for box in boxes:
    # check if any boxes are in a corner and cannot be moved by keeper
    # if so, return infinity (prune tree)
    if corner_check(s,box[0],box[1]):
        return float('inf')
    # calculate Manhattan distance to each goal, save the min one
    total_cost += min(abs(box[0]-star[0]) + abs(box[1]-star[1]) for star in stars)

  # also add the distance between keeper and closest box
  krow, kcol = getKeeperPosition(s)
  total_cost += min(abs(krow-box[0]) + abs(kcol-box[1]) for box in boxes)

  return total_cost

def corner_check(s,r,c):
  '''
  A function that, given a state, s, and box position, checks if a box is stuck
  in a corner and thus can never be pushed to a goal.

  Inputs:
  - s, numpy array, state of the game
  - r, integer, row of box
  - c, integer, col of box

  Returns:
  - boolean, True if the box is stuck in a corner, and False otherwise

  '''
  # check if walls on any of its four sides
  wall_above = isWall(get_square(s,r-1,c))
  wall_below = isWall(get_square(s,r+1,c))
  wall_left = isWall(get_square(s,r,c-1))
  wall_right = isWall(get_square(s,r,c+1))

  # if there is a wall on two sides - stuck in a corner return True
  if wall_above and wall_left or wall_above and wall_right or wall_below and wall_left or wall_below and wall_right:
    return True

  # not in a corner
  return False

# Some predefined problems with initial state s (array). Sokoban function will automatically transform it to numpy
# array. For other function, the state s is presented as a numpy array. You can just call sokoban(init-state,
# heuristic function) to test the result Each problem can be visualized by calling prettyMoves(path) and printlists(
# path) in a_star function
#
# Problems are roughly ordered by their difficulties.
# For most problems, we also provide 2 additional number per problem:
#    1) # of nodes expanded by A* using our next-states and h0 heuristic.
#    2) the depth of the optimal solution.
# These numbers are located at the comments of the problems. For example, the first problem below
# was solved by 80 nodes expansion of A* and its optimal solution depth is 7.
#
# Your implementation may not result in the same number of nodes expanded, but it should probably
# give something in the same ballpark. As for the solution depth, any admissible heuristic must
# make A* return an optimal solution. So, the depths of the optimal solutions provided could be used
# for checking whether your heuristic is admissible.
#
# Warning: some problems toward the end are quite hard and could be impossible to solve without a good heuristic!


# [80,7]
s1 = [[1, 1, 1, 1, 1, 1],
      [1, 0, 3, 0, 0, 1],
      [1, 0, 2, 0, 0, 1],
      [1, 1, 0, 1, 1, 1],
      [1, 0, 0, 0, 0, 1],
      [1, 0, 0, 0, 4, 1],
      [1, 1, 1, 1, 1, 1]]

# [110,10],
s2 = [[1, 1, 1, 1, 1, 1, 1],
      [1, 0, 0, 0, 0, 0, 1],
      [1, 0, 0, 0, 0, 0, 1],
      [1, 0, 0, 2, 1, 4, 1],
      [1, 3, 0, 0, 1, 0, 1],
      [1, 1, 1, 1, 1, 1, 1]]

# [211,12],
s3 = [[1, 1, 1, 1, 1, 1, 1, 1, 1],
      [1, 0, 0, 0, 1, 0, 0, 0, 1],
      [1, 0, 0, 0, 2, 0, 3, 4, 1],
      [1, 0, 0, 0, 1, 0, 0, 0, 1],
      [1, 0, 0, 0, 1, 0, 0, 0, 1],
      [1, 1, 1, 1, 1, 1, 1, 1, 1]]

# [300,13],
s4 = [[1, 1, 1, 1, 1, 1, 1],
      [0, 0, 0, 0, 0, 1, 4],
      [0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 1, 1, 0, 0],
      [0, 0, 1, 0, 0, 0, 0],
      [0, 2, 1, 0, 0, 0, 0],
      [0, 3, 1, 0, 0, 0, 0]]

# [551,10],
s5 = [[1, 1, 1, 1, 1, 1],
      [1, 1, 0, 0, 1, 1],
      [1, 0, 0, 0, 0, 1],
      [1, 4, 2, 2, 4, 1],
      [1, 0, 0, 0, 0, 1],
      [1, 1, 3, 1, 1, 1],
      [1, 1, 1, 1, 1, 1]]

# [722,12],
s6 = [[1, 1, 1, 1, 1, 1, 1, 1],
      [1, 0, 0, 0, 0, 0, 4, 1],
      [1, 0, 0, 0, 2, 2, 3, 1],
      [1, 0, 0, 1, 0, 0, 4, 1],
      [1, 1, 1, 1, 1, 1, 1, 1]]

# [1738,50],
s7 = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
      [0, 0, 1, 1, 1, 1, 0, 0, 0, 3],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
      [0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
      [0, 2, 1, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 1, 4]]

# [1763,22],
s8 = [[1, 1, 1, 1, 1, 1],
      [1, 4, 0, 0, 4, 1],
      [1, 0, 2, 2, 0, 1],
      [1, 2, 0, 1, 0, 1],
      [1, 3, 0, 0, 4, 1],
      [1, 1, 1, 1, 1, 1]]

# [1806,41],
s9 = [[1, 1, 1, 1, 1, 1, 1, 1, 1],
      [1, 1, 1, 0, 0, 1, 1, 1, 1],
      [1, 0, 0, 0, 0, 0, 2, 0, 1],
      [1, 0, 1, 0, 0, 1, 2, 0, 1],
      [1, 0, 4, 0, 4, 1, 3, 0, 1],
      [1, 1, 1, 1, 1, 1, 1, 1, 1]]

# [10082,51],
s10 = [[1, 1, 1, 1, 1, 0, 0],
       [1, 0, 0, 0, 1, 1, 0],
       [1, 3, 2, 0, 0, 1, 1],
       [1, 1, 0, 2, 0, 0, 1],
       [0, 1, 1, 0, 2, 0, 1],
       [0, 0, 1, 1, 0, 0, 1],
       [0, 0, 0, 1, 1, 4, 1],
       [0, 0, 0, 0, 1, 4, 1],
       [0, 0, 0, 0, 1, 4, 1],
       [0, 0, 0, 0, 1, 1, 1]]

# [16517,48],
s11 = [[1, 1, 1, 1, 1, 1, 1],
       [1, 4, 0, 0, 0, 4, 1],
       [1, 0, 2, 2, 1, 0, 1],
       [1, 0, 2, 0, 1, 3, 1],
       [1, 1, 2, 0, 1, 0, 1],
       [1, 4, 0, 0, 4, 0, 1],
       [1, 1, 1, 1, 1, 1, 1]]

# [22035,38],
s12 = [[0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0],
       [1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
       [1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1],
       [1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 2, 1, 1, 1, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 1, 0, 1, 4, 0, 4, 1],
       [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]]

# [26905,28],
s13 = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 4, 0, 0, 0, 0, 0, 2, 0, 1],
       [1, 0, 2, 0, 0, 0, 0, 0, 4, 1],
       [1, 0, 3, 0, 0, 0, 0, 0, 2, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 4, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

# [41715,53],
s14 = [[0, 0, 1, 0, 0, 0, 0],
       [0, 2, 1, 4, 0, 0, 0],
       [0, 2, 0, 4, 0, 0, 0],
       [3, 2, 1, 1, 1, 0, 0],
       [0, 0, 1, 4, 0, 0, 0]]

# [48695,44],
s15 = [[1, 1, 1, 1, 1, 1, 1],
       [1, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 2, 2, 0, 1],
       [1, 0, 2, 0, 2, 3, 1],
       [1, 4, 4, 1, 1, 1, 1],
       [1, 4, 4, 1, 0, 0, 0],
       [1, 1, 1, 1, 0, 0, 0]]

# [91344,111],
s16 = [[1, 1, 1, 1, 1, 0, 0, 0],
       [1, 0, 0, 0, 1, 0, 0, 0],
       [1, 2, 1, 0, 1, 1, 1, 1],
       [1, 4, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 5, 0, 5, 0, 1],
       [1, 0, 5, 0, 1, 0, 1, 1],
       [1, 1, 1, 0, 3, 0, 1, 0],
       [0, 0, 1, 1, 1, 1, 1, 0]]

# [3301278,76],
# Warning: This problem is very hard and could be impossible to solve without a good heuristic!
s17 = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 3, 0, 0, 1, 0, 0, 0, 4, 1],
       [1, 0, 2, 0, 2, 0, 0, 4, 4, 1],
       [1, 0, 2, 2, 2, 1, 1, 4, 4, 1],
       [1, 0, 0, 0, 0, 1, 1, 4, 4, 1],
       [1, 1, 1, 1, 1, 1, 0, 0, 0, 0]]

# [??,25],
s18 = [[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 4, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 4, 1, 0, 0, 0, 0]]

# [??,21],
s19 = [[0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 2, 0],
       [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 4],
       [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 2, 0, 4, 1, 0, 0, 0]]


# Utility functions for printing states and moves.
# You do not need to understand any of the functions below this point.


# Helper function of prettyMoves
# Detect the move from state s --> s1
def detectDiff(s, s1):
    row, col = getKeeperPosition(s)
    row1, col1 = getKeeperPosition(s1)
    if (row1 == row + 1):
        return 'Down'
    if (row1 == row - 1):
        return 'Up'
    if (col1 == col + 1):
        return 'Right'
    if (col1 == col - 1):
        return 'Left'
    return 'fail'


# Translates a list of states into a list of moves
def prettyMoves(lists):
    initial = 0
    action = []
    for states in (lists):
        if (initial != 0):
            action.append(detectDiff(previous, states))
        initial = 1
        previous = states
    return action


# Print the content of the square to stdout.
def printsquare(v):
    if (v == blank):
        print(' ', end='')
    if (v == wall):
        print('#', end='')
    if (v == box):
        print('$', end='')
    if (v == keeper):
        print('@', end='')
    if (v == star):
        print('.', end='')
    if (v == boxstar):
        print('*', end='')
    if (v == keeperstar):
        print('+', end='')


# Print a state
def printstate(s):
    row = s.shape[0]
    col = s.shape[1]
    for i in range(row):
        for j in range(col):
            printsquare(s[i, j])
        print('\n')


# Print a list of states with delay.
def printlists(lists):
    for states in (lists):
        printstate(states)
        print('\n')


if __name__ == "__main__":
    sokoban(s1, h0)

    sokoban(s2, h0)

    sokoban(s3, h0)

    sokoban(s4, h0)


Nodes Generated by A*: 190
Nodes Expanded by A*: 74
Solution Depth: 7
Nodes Generated by A*: 361
Nodes Expanded by A*: 126
Solution Depth: 10
Nodes Generated by A*: 634
Nodes Expanded by A*: 220
Solution Depth: 12
Nodes Generated by A*: 812
Nodes Expanded by A*: 287
Solution Depth: 13
